<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/baseline/notebooks/05_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Ensemble and Submission


## Models

- **BiLSTM:** 3 layers, hidden size 256, ByteLevel BPE, Adam, LR 1e-3, epoch 3
- **BiGRU:** 3 layers, hidden size 256, Whitespace BPE, Adam, LR 1e-3, epoch 3

Both use `viterbi_lines` decoding

## Environment Setup

In [68]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)

/content/UA-Locations-NER-Deep-Learning


In [69]:
%pip install -q -r "$repo_dir/requirements.txt"

In [70]:
import json

import pandas as pd
from google.colab import drive

from src.data_io import get_data_dir
from src.metrics import comp_metric, hard_processing, location_errors
from src.postprocessing import postprocess

## Load data

In [71]:
drive.mount("/content/drive", force_remount=True)

RESULTS_DIR = Path("/content/drive/MyDrive/ner_results/shared")
MODEL_DIRS = {"bilstm": RESULTS_DIR / "bilstm/bilstm_3l_h256_b128_s17_retrain_e3_viterbi_lines", "bigru": RESULTS_DIR / "bigru/bigru_3l_h256_adam_lr1e-3_e3_viterbi_lines"}

def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

VAL_GOLD_PATH = RESULTS_DIR / "val_gold.jsonl"

val_gold = load_jsonl(VAL_GOLD_PATH)
val_predictions = {name: load_jsonl(folder / "val.jsonl") for name, folder in MODEL_DIRS.items()}
test_predictions = {name: load_jsonl(folder / "test.jsonl") for name, folder in MODEL_DIRS.items()}

Mounted at /content/drive


Fixing rows order:

In [72]:
gold_ids = [row["text_id"] for row in val_gold]

for name, rows in val_predictions.items():
    by_id = {row["text_id"]: row for row in rows}

    if len(by_id) != len(rows):
        raise ValueError(f"{name}: duplicate ID")
    if set(by_id) != set(gold_ids):
        raise ValueError(f"{name}: lost or extra ID")

    rows = [by_id[text_id] for text_id in gold_ids]

    for gold, prediction in zip(val_gold, rows):
        if gold["text"] != prediction["text"]:
            raise ValueError(f"{name}: texts do not match")

    val_predictions[name] = rows

Validation spans:

In [73]:
for rows in [val_gold, *val_predictions.values()]:
    for row in rows:
        for start, end in row["spans"]:
            if not 0 <= start < end <= len(row["text"]):
                raise ValueError(f"Incorrect span: {row['text_id']}")

## Baseline metrics

In [74]:
CLEAN_OPTIONS = {"expand": True, "trim": True, "empty": True}

y_true = []
for row in val_gold:
    y_true.append([row["text"][a:b] for a, b in row["spans"]])

val_locations = {}
for name, rows in val_predictions.items():
    predictions = []
    for row in rows:
        locations = postprocess(row["text"], row["spans"], **CLEAN_OPTIONS)
        predictions.append(locations)
    val_locations[name] = predictions

baseline_scores = {}
for name, predictions in val_locations.items():
    baseline_scores[name] = comp_metric(y_true, predictions)

pd.Series(baseline_scores, name="validation_f1").to_frame()

,validation_f1
bilstm,0.896171
bigru,0.883126


### Local validation vs. Kaggle

Public leaderboard scores for the corresponding model submissions:

| Model | Public F1 |
|---|---:|
| BiGRU, Whitespace BPE, epoch 3, Viterbi lines | 0.59084 |
| BiLSTM, 3 layers, hidden 256, Viterbi lines | 0.57383 |

Local validation scores differ from public leaderboard results.


In [75]:
def additional_metrics(gold, predictions):
    tp = fp = fn = 0
    text_f1 = []
    empty_texts = empty_errors = 0

    for truth, guess in zip(gold, predictions):
        truth = set(map(hard_processing, truth))
        guess = set(map(hard_processing, guess))

        matched = len(truth & guess)
        tp += matched
        fp += len(guess - truth)
        fn += len(truth - guess)

        total = len(truth) + len(guess)
        text_f1.append(2 * matched / total if total else 1.0)

        if not truth:
            empty_texts += 1
            empty_errors += bool(guess)

    return {
        "official_f1": comp_metric(gold, predictions),
        "precision": tp / (tp + fp) if tp + fp else float(not fn),
        "recall": tp / (tp + fn) if tp + fn else 1.0,
        "mean_text_f1": sum(text_f1) / len(text_f1),
        "empty_fp_rate": empty_errors / empty_texts if empty_texts else None,
        "FP": fp,
        "FN": fn,
    }

In [76]:
metric_rows = []

for name, predictions in val_locations.items():
    metrics = additional_metrics(y_true, predictions)
    metric_rows.append({"model": name, **metrics})

pd.DataFrame(metric_rows).set_index("model").round(6)

,official_f1,precision,recall,mean_text_f1,empty_fp_rate,FP,FN
model,,,,,,,
bilstm,0.896171,0.903261,0.889191,0.962671,0.014774,8730,10158
bigru,0.883126,0.895623,0.870973,0.958177,0.014371,9305,11828


BiLSTM makes fewer FP (8,480 vs. 9,516) and FN (9,599 vs. 11,451), resulting in higher precision and recall.

## Diversity & errors

In [77]:
def normalize_set(locations):
    return {hard_processing(location) for location in locations}

gold_sets = list(map(normalize_set, y_true))
model_sets = {}
for name, predictions in val_locations.items():
    model_sets[name] = list(map(normalize_set, predictions))

In [78]:
comparison = {
    "both_right": 0,
    "only_bilstm_right": 0,
    "only_bigru_right": 0,
    "both_wrong": 0,
}

for gold, lstm, gru in zip(gold_sets, model_sets["bilstm"], model_sets["bigru"]):
    if lstm == gold and gru == gold:
        comparison["both_right"] += 1
    elif lstm == gold:
        comparison["only_bilstm_right"] += 1
    elif gru == gold:
        comparison["only_bigru_right"] += 1
    else:
        comparison["both_wrong"] += 1

pd.Series(comparison, name="texts").to_frame()

,texts
both_right,233004
only_bilstm_right,5306
only_bigru_right,3751
both_wrong,8806


Both models fail on 8,533 texts. Their errors overlap only partly, which gives potential for ensembling (but still it is not a guarantee of F1 improvement).

In [79]:
examples = []

for i, row in enumerate(val_gold):
    lstm = val_locations["bilstm"][i]
    gru = val_locations["bigru"][i]

    if model_sets["bilstm"][i] == gold_sets[i] and model_sets["bigru"][i] == gold_sets[i]:
        continue

    examples.append({
        "text_id": row["text_id"],
        "text": row["text"],
        "gold": y_true[i],
        "bilstm": lstm,
        "bilstm_errors": location_errors(y_true[i], lstm),
        "bigru": gru,
        "bigru_errors": location_errors(y_true[i], gru)})

    if len(examples) == 5:
        break

display(pd.DataFrame(examples))

,text_id,text,gold,bilstm,bilstm_errors,bigru,bigru_errors
0,uk_79,Елі Кара та Уріель Кельман вилетіли до Гонконг...,[Гонконга одра],[Гонконга],"{'false_positives': ['гонконга'], 'false_negat...",[Гонконга],"{'false_positives': ['гонконга'], 'false_negat..."
1,uk_96,Незабаром придбають і 10 тролейбусів зі збільш...,[Кременчука],[],"{'false_positives': [], 'false_negatives': ['к...",[Кременчука],"{'false_positives': [], 'false_negatives': []}"
2,uk_151,Вони бурхливо обговорювали це в загальному чат...,[Мессі],[],"{'false_positives': [], 'false_negatives': ['м...",[],"{'false_positives': [], 'false_negatives': ['м..."
3,uk_390,"Своєю чергою, Путін заявив, що роботи з прокла...","[Турецького потоку, Угорщині, Будапешт]","[Турецького потоку, Угорщині, Будапешт]","{'false_positives': [], 'false_negatives': []}","[Угорщині, Будапешт]","{'false_positives': [], 'false_negatives': ['т..."
4,uk_609,"У Росії чоловік стверджує, що iPhone ""перетвор...",[Росії],"[Росії, Світ]","{'false_positives': ['світ'], 'false_negatives...","[Росії, Світ]","{'false_positives': ['світ'], 'false_negatives..."


Error correlation:

In [80]:
errors = {}
# text error means at least one FP or FN
for name, predictions in model_sets.items():
    errors[name] = []
    for gold, predicted in zip(gold_sets, predictions):
        errors[name].append(int(predicted != gold))

errors = pd.DataFrame(errors)
print(f"BiLSTM text error rate: {errors['bilstm'].mean():.2%}")
print(f"BiGRU text error rate: {errors['bigru'].mean():.2%}")
print(f"Error correlation: {errors['bilstm'].corr(errors['bigru']):.3f}")

BiLSTM text error rate: 5.01%
BiGRU text error rate: 5.63%
Error correlation: 0.643


### Conclusions:

- BiLSTM leads on VAL in F1, precision and recall, with fewer FP/FN. BiGRU leads on the public leaderboard.
- Error correlation is 0.621. Each model correctly handles texts the other misses, suggesting potential for ensembling.
- We compare ensemble variants using the same postprocessing and track both F1 and FP/FN.

## Ensembling
Union vs. Intersection

- **Union:** keep locations predicted by either model - favors recall.
- **Intersection:** keep only locations predicted by both models - favors precision.

In [81]:
def combine_predictions(predictions, method):
    result = []

    for rows in zip(*predictions.values()):
        location_sets = list(map(normalize_set, rows))
        if method == "union":
            selected = set.union(*location_sets)
        elif method == "intersection":
            selected = set.intersection(*location_sets)
        else:
            raise ValueError(method)

        originals = {}
        for locations in rows:
            for location in locations:
                originals.setdefault(hard_processing(location), location)
        result.append([originals[key] for key in sorted(selected)])

    return result


val_ensembles = {
    "union": combine_predictions(val_locations, "union"),
    "intersection": combine_predictions(val_locations, "intersection")
}

In [82]:
scores = baseline_scores.copy()
for name, predictions in val_ensembles.items():
    scores[name] = comp_metric(y_true, predictions)

pd.Series(scores, name="validation_f1").to_frame()

,validation_f1
bilstm,0.896171
bigru,0.883126
union,0.887864
intersection,0.891627


In [83]:
candidates = {**val_locations, **val_ensembles}
reference_name = max(baseline_scores, key=baseline_scores.get)
reference = val_locations[reference_name]

metric_rows = []

for name, predictions in candidates.items():
    metrics = additional_metrics(y_true, predictions)
    fixed = new = 0

    for gold, before, after in zip(y_true, reference, predictions):
        before_errors = location_errors(gold, before)
        after_errors = location_errors(gold, after)

        for kind in ["false_positives", "false_negatives"]:
            old = set(before_errors[kind])
            current = set(after_errors[kind])
            fixed += len(old - current)
            new += len(current - old)

    metric_rows.append({
        "model": name,
        **metrics,
        "fixed_errors": fixed,
        "new_errors": new,
    })

print("Reference model:", reference_name)
display(pd.DataFrame(metric_rows).set_index("model").round(6))

Reference model: bilstm


,official_f1,precision,recall,mean_text_f1,empty_fp_rate,FP,FN,fixed_errors,new_errors
model,,,,,,,,,
bilstm,0.896171,0.903261,0.889191,0.962671,0.014774,8730,10158,0,0
bigru,0.883126,0.895623,0.870973,0.958177,0.014371,9305,11828,6317,8562
union,0.887864,0.862892,0.914324,0.960494,0.021495,13318,7854,2304,4588
intersection,0.891627,0.942655,0.845840,0.959678,0.007018,4717,14132,4013,3974


**Findings:**
- Union increases recall to 0.925 but adds FP, reducing precision to 0.857.
- Intersection raises precision to 0.953 but adds FN, lowering recall to 0.846.
- Neither improves VAL F1 over BiLSTM, but intersection comes closest (0.8962 vs. 0.9008).

In [84]:
groups = {"UK": [], ">150 chars": [], "UK, >150 chars": [], "2+ gold locations": []}
for i, row in enumerate(val_gold):
    is_uk = row["text_id"].startswith("uk_")
    is_long = len(row["text"]) > 150

    if is_uk:
        groups["UK"].append(i)
    if is_long:
        groups[">150 chars"].append(i)
    if is_uk and is_long:
        groups["UK, >150 chars"].append(i)
    if len(gold_sets[i]) >= 2:
        groups["2+ gold locations"].append(i)

results = []
for group, indices in groups.items():
    gold = [y_true[i] for i in indices]

    for name, predictions in candidates.items():
        selected = [predictions[i] for i in indices]
        metrics = additional_metrics(gold, selected)
        results.append({"group": group, "model": name, "F1": metrics["official_f1"],
                        "precision": metrics["precision"], "recall": metrics["recall"]})

display(pd.DataFrame(results).set_index(["group", "model"]).round(4))

F1  precision  recall
group             model                                  
UK                bilstm        0.8866     0.8941  0.8792
                  bigru         0.8755     0.8891  0.8622
                  union         0.8784     0.8547  0.9035
                  intersection  0.8839     0.9352  0.8380
>150 chars        bilstm        0.8932     0.8992  0.8873
                  bigru         0.8801     0.8910  0.8695
                  union         0.8848     0.8575  0.9139
                  intersection  0.8887     0.9398  0.8429
UK, >150 chars    bilstm        0.8794     0.8869  0.8720
                  bigru         0.8685     0.8809  0.8565
                  union         0.8711     0.8459  0.8979
                  intersection  0.8771     0.9291  0.8305
2+ gold locations bilstm        0.8985     0.9266  0.8721
                  bigru         0.8877     0.9197  0.8578
                  union         0.8998     0.8999  0.8997
                  intersection  0.8860     0.9498  0.8303

**Findings:**

- BiLSTM leads on UK and longer texts; intersection is close but misses more locations.
- For texts with 2+ locations, union matches BiLSTM F1 with higher recall and lower precision.


## Submission

In [85]:
data_dir = get_data_dir(ENVIRONMENT)
test = pd.read_csv(data_dir / "test.csv")
test_ids = test["text_id"].tolist()

for name, rows in test_predictions.items():
    by_id = {row["text_id"]: row for row in rows}

    if len(by_id) != len(rows):
        raise ValueError(f"{name}: duplicate test ID")
    if set(by_id) != set(test_ids):
        raise ValueError(f"{name}: missing or extra test ID")

    rows = [by_id[text_id] for text_id in test_ids]

    for text, row in zip(test["text"], rows):
        if text != row["text"]:
            raise ValueError(f"{name}: test texts do not match")

        for start, end in row["spans"]:
            if not 0 <= start < end <= len(text):
                raise ValueError(f"{name}: incorrect test span")

    test_predictions[name] = rows

In [86]:
test_locations = {}

for name, rows in test_predictions.items():
    locations = []
    for row in rows:
        locations.append(postprocess(row["text"], row["spans"], **CLEAN_OPTIONS))
    test_locations[name] = locations

In [89]:
selected = combine_predictions(test_locations, "intersection")

submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(locations, ensure_ascii=False)
    for locations in selected
]

output_path = RESULTS_DIR / "submission_intersection_original_cleanup.csv"
submission.to_csv(output_path, index=False, mode="x")

print(output_path)
display(submission.head())

/content/drive/MyDrive/ner_results/shared/submission_intersection_original_cleanup.csv


,text_id,locations
0,0,[]
1,1,"[""Києві"", ""Шулявського шляхопроводу""]"
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,[]


## Another cleanup

Removes punctuation and emoji components only at span edges, preserving characters inside names. Word expansion and empty filtering remain the same. Applied to both models before intersection.

In [90]:
import unicodedata
from src.postprocessing import EDGE_CHARS, restore_word


def edge_cleanup(text, spans, expand=False, trim=False, empty=False):
    def edge_symbol(char):
        return (
            char in EDGE_CHARS
            or unicodedata.category(char) == "So"
            or char == "\u200d"
            or "\ufe00" <= char <= "\ufe0f"
            or "\U0001f3fb" <= char <= "\U0001f3ff"
            or "\U000e0100" <= char <= "\U000e01ef"
        )

    locations = []
    for start, end in spans:
        if expand and sum(char.isalpha() for char in text[start:end]) > 4:
            start, end = restore_word(text, start, end)
        if trim:
            while start < end and edge_symbol(text[start]):
                start += 1
            while end > start and edge_symbol(text[end - 1]):
                end -= 1

        location = text[start:end]

        if not any(char.isalnum() for char in location):
            continue
        if empty and not hard_processing(location):
            continue
        locations.append(location)

    return locations

In [91]:
EDGE_OPTIONS = CLEAN_OPTIONS.copy()
print("Options:", EDGE_OPTIONS)

edge_val_locations = {}

for name, rows in val_predictions.items():
    locations = []
    for row in rows:
        locations.append(edge_cleanup(row["text"], row["spans"], **EDGE_OPTIONS))
    edge_val_locations[name] = locations

old_intersection = combine_predictions(val_locations, "intersection")
edge_intersection = combine_predictions(edge_val_locations, "intersection")

comparison = []
for name, predictions in [("original cleanup", old_intersection), ("edge cleanup", edge_intersection)]:
    metrics = additional_metrics(y_true, predictions)
    comparison.append({"cleanup": name, "F1": metrics["official_f1"], "precision": metrics["precision"],
                       "recall": metrics["recall"], "FP": metrics["FP"], "FN": metrics["FN"]})

display(pd.DataFrame(comparison).set_index("cleanup").round(6))

Options: {'expand': True, 'trim': True, 'empty': True}


,F1,precision,recall,FP,FN
cleanup,,,,,
original cleanup,0.891627,0.942655,0.845840,4717,14132
edge cleanup,0.891649,0.942690,0.845851,4714,14131


Edge cleanup leaves VAL performance almost unchanged, with -3 FP and -1 FN:

In [92]:
edge_test_locations = {}

for name, rows in test_predictions.items():
    locations = []
    for row in rows:
        locations.append(edge_cleanup(row["text"], row["spans"], **EDGE_OPTIONS))
    edge_test_locations[name] = locations

edge_selected = combine_predictions(edge_test_locations, "intersection")

for text, locations in zip(test["text"], edge_selected):
    for location in locations:
        if location not in text:
            raise ValueError(f"Not an original substring: {location!r}")

edge_submission = test[["text_id"]].copy()
edge_submission["locations"] = [
    json.dumps(locations, ensure_ascii=False)
    for locations in edge_selected
]

output_path = RESULTS_DIR / "submission_intersection_edge_cleanup.csv"
edge_submission.to_csv(output_path, index=False, mode="x")

print(output_path)
display(edge_submission.head())

/content/drive/MyDrive/ner_results/shared/submission_intersection_edge_cleanup.csv


,text_id,locations
0,0,[]
1,1,"[""Києві"", ""Шулявського шляхопроводу""]"
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,[]


## Public Scores

| Ensemble | Cleanup | Public F1 |
|---|---|---:|
| Union | Original | 0.56549 |
| Intersection | Original | 0.60645 |
| Intersection | Edge | **0.60869** |

Intersection outperforms union on the public leaderboard. Edge cleanup adds 0.00224 F1 - our best score.